# Analyse Mt. Kenya data
Follow this procedure to analyse the level2 data

### CAMS data
The part about CAMS data needs to be run only once and downloads the CAMS data to your local computer (taking a long time). 

It requires to have a login at the [Atmosphere data store](https://ads.atmosphere.copernicus.eu/cdsapp#!/home) and an installation of [cdsapi](https://cds.climate.copernicus.eu/api-how-to). 

This data is then read in and saved as netcdfs in the `./data/cams/` folder. 

If this was already done, this first part can be skipped. 


In [1]:
import numpy as np
import os,sys
import xarray as xr
import pandas as pd

import matplotlib.pyplot as plt

%load_ext autoreload
%matplotlib widget

## Add parent directory to syspath
parent_dir = os.path.abspath(os.path.join(os.path.dirname('.'), '..'))
if not parent_dir in sys.path:
    sys.path.append(parent_dir)

In [2]:
## general settings
stat = 'MKN'

dir_data_cams = r"..\..\..\Data\CAMS" #adapt path to local folder to save CAMS data

### 1) Get and read CAMS data

In [ ]:
## Get CAMS data from datastore
import get_cams #require to install cdsapi (https://cds.climate.copernicus.eu/api-how-to)

## Get CAMS data
# This may take several days, and it is only required to be done once
# It was done in Jan 2024, and the data will be saved in seperate netcdf files (see next step)
# TODO wrong path when running here??
get_cams.main(dir_data=dir_data_cams + r"\CAMS", yr1=2003, yr2=2020,which='cams_egg4',station=[stat])
get_cams.main(dir_data=dir_data_cams + r"\CAMS", yr1=2003, yr2=2022,which='cams_eac4',station=[stat])
get_cams.main(dir_data=dir_data_cams + r"\CAMS", yr1=2020, yr2=2023,which='cams_inv_co2',station=[stat])
get_cams.main(dir_data=dir_data_cams + r"\CAMS", yr1=2020, yr2=2021,which='cams_inv_ch4',station=[stat])
get_cams.main(dir_data=dir_data_cams + r"\CAMS", yr1=2020, yr2=2023,which='cams_gfas',station=[stat])

In [ ]:
### Save CAMS data as netcdf
import analyses_level2.read_cams as read_cams

# Read in the CAMS data and save as netcdfs (one for each cams-dataset)
# The netcdfs will be saved in .\data\cams\

#those take about a minute:
read_cams.read_cams_inv(dir_data_cams,species='co2',yr1=2020,yr2=2023,station=stat)
read_cams.read_cams_inv(dir_data_cams,species='ch4',yr1=2020,yr2=2021,dx=3,dy=2,fact_dxy=2,station=stat) #ch4: coarser resolution

#those are going faster: 
read_cams.read_cams_eac4(dir_data_cams,station=stat)
read_cams.read_cams_egg4(dir_data_cams,yr1=2003,yr2=2020,station=stat)
read_cams.read_cams_gfas(dir_data_cams)

In [ ]:
### Read CAMS netcdfs

### 2) Read in the gaw kenya data
All the data has to be saved in `../data/` (or another folder given in data_path).\
All data has to be indicated in the dictionary AvailableData (defined in `read_data.py`). \
If you add new data, please add also an entry to this dictionary (the entry `dataset` has to be a unique name. )

In [ ]:
## Read all data
%autoreload 2
from analyses_level2.read_data import AvailableData, create_data_reader

# File path
data_path = "../data/"

# if New data is added to ./data folder, adapt the dictionary in AvailableData
all_data = list(AvailableData)
print(all_data)

#####---------- TO ADAPT ---------------#####
selected_data = ['CO2', 'CO2_flask', 
                 'CO', 'CO_flask', 
                 'CH4', 'CH4_flask', 
                 'O3'
                 ] # define data to read in. If empty, all data is used 
## 
processing_kwargs = { 
    'FLASK_FLAG_CORR' : True # exclude flagged flask-data
}
#####-----------------------------------#####

datasets = [] # initialize list of all datasets 
# read in data
for sel in (selected_data if selected_data else all_data):
    #define where the data has to be read from
    data_reader =  create_data_reader(data_path=data_path,dataset=sel,**processing_kwargs) #creates an instance of the desired data_reader class
    print(f"Data from {data_reader.__class__.__name__} for {sel}:")

    # call the data-reading function on that instance: 
    data = data_reader.read_data() 
    # call the data-processing
    data = data_reader.process_data(data)

    # prepare merged dataset
    data = data.drop(columns='endtime') # problem when merging datasets (because of NaT?), so better remove endtime
    ds = data.to_xarray()
    ds = ds.assign_coords(dataset=sel)
    ds['species'] = data_reader.species
    ds['unit']  = np.unique(ds.unit.dropna(dim='time'))[0]
    datasets.append(ds)

# save all in one xarray dataset
ds_all = xr.concat(datasets,dim="dataset")


### 3) Compare to CAMS
Select for each dataset the best-correlated cams grid. 
Save that cams-data as a seperate netcdf


In [ ]:
import read_cams

## this takes some time (need to do it once, then just load the nc file in the next cell)
cams_best_grid = read_cams.get_best_cams(ds_all,save_netcdf=True)

In [ ]:
#once it is saved, it can be loaded with: 
cams_best_grid = xr.open_dataset(f"{data_path}/cams/cams_best_grid_MKN.nc")